In [5]:
import keras
import numpy as np
import tensorflow as tf
#创建训练数据集
shakespeare_url = "https://homl.info/shakespeare" # shortcut RL 
filepath = keras.utils.get_file("shakespeare.txt", shakespeare_url) 
with open(filepath) as f: 
    shakespeare_text = f.read() 

#为文本添加一个分词器：它会找到文本中使用的所有字符，并将它们映射到不同的字符ID，从1到不同字符的数量
tokenizer = keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts([shakespeare_text])

max_id = len(tokenizer.word_index)
# 对全文进行编码，以便每个字符都由其ID表示
[encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1 
dataset_size = len(encoded)
#使用文本的前90%作为训练集,，并创建一个t.data.Dataset，它将从该集合中逐个返回每个字符
train_size = dataset_size * 90 // 100 
dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size]) 

tokenizer.texts_to_sequences(["First"]), tokenizer.sequences_to_texts([[20, 6, 9, 8, 3]]) 

([[20, 6, 9, 8, 3]], ['f i r s t'])

In [6]:
#将顺序数据集切成多个窗口
n_steps = 100 
window_length = n_steps + 1 # target = input shifted 1 character ahead 
dataset = dataset.window(window_length, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(window_length))
batch_size = 32 
dataset = dataset.shuffle(10000).batch(batch_size) 
dataset = dataset.map(lambda windows: (windows[:, :-1], windows[:, 1:]))
dataset = dataset.map( 
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=max_id), Y_batch)) 
dataset = dataset.prefetch(1)


In [7]:
import tensorflow as tf

# 1. 检查 dataset 类型和大致结构
print(f"Dataset type: {type(dataset)}")

# 2. 尝试从 dataset 中取出一个批次进行检查
try:
    # 从 dataset 中取一个批次
    for example_batch in dataset.take(1):
        break # 只取第一个批次就跳出
    
    # 3. 分析这个批次的结构和内容
    if isinstance(example_batch, tuple) and len(example_batch) == 2:
        inputs, targets = example_batch
        print(f"Input shape (should be [batch_size, seq_len, max_id]): {inputs.shape}")
        print(f"Input dtype: {inputs.dtype}")
        print(f"Target shape (should be [batch_size, seq_len]): {targets.shape}")
        print(f"Target dtype: {targets.dtype}")
        print(f"Sample input (first item, first timestep): {inputs[0][0].numpy()}")
        print(f"Sample target (first item, first timestep): {targets[0][0].numpy()}")
        
        # 4. 检查 targets 的最大值是否小于 max_id ()
        max_target_value = tf.reduce_max(targets).numpy()
        print(f"Maximum value in targets: {max_target_value}")
        print(f"max_id (should be > max_target_value): {max_id}")
        if max_target_value >= max_id:
            print(f"ERROR: Max target value ({max_target_value}) is >= max_id ({max_id}). This will cause an error during loss calculation.")
        
    else:
        print(f"ERROR: Dataset batch structure is unexpected: {example_batch}. Expected (inputs, targets) tuple.")
        print(f"Full batch structure: {example_batch}")

except Exception as e:
    print(f"Error when trying to access dataset: {e}")

# 5. 确保 max_id 是正确的数值类型 (int 或 tf.Tensor/int)
print(f"Type of max_id: {type(max_id)}, Value: {max_id}")

# --- 只有在上述检查都通过后，才运行 model.fit ---
# history = model.fit(dataset, epochs=20)

Dataset type: <class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
Input shape (should be [batch_size, seq_len, max_id]): (32, 100, 39)
Input dtype: <dtype: 'float32'>
Target shape (should be [batch_size, seq_len]): (32, 100)
Target dtype: <dtype: 'int32'>
Sample input (first item, first timestep): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Sample target (first item, first timestep): 15
Maximum value in targets: 35
max_id (should be > max_target_value): 39
Type of max_id: <class 'int'>, Value: 39


In [8]:
#创建和训练Char-RNN模型
model = keras.models.Sequential([ 
    keras.layers.GRU(128, return_sequences=True, input_shape=[None, max_id], 
                     dropout=0.2, recurrent_dropout=0.2), 
    keras.layers.GRU(128, return_sequences=True, 
                     dropout=0.2, recurrent_dropout=0.2), 
    keras.layers.TimeDistributed(keras.layers.Dense(max_id, 
                     activation="softmax")) 
]) 
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam") 
history = model.fit(dataset, epochs=20)

Epoch 1/20
   2730/Unknown - 398s 144ms/step - loss: 1.8100

KeyboardInterrupt: 

In [12]:
def preprocess(texts): 
    X = np.array(tokenizer.texts_to_sequences(texts)) - 1 
    return tf.one_hot(X, max_id)

X_new = preprocess(["How are yo"]) 
Y_proba = model.predict(X_new)
# print("Y_proba shape:", Y_proba.shape) # 检查预测概率的形状
Y_pred = np.argmax(Y_proba, axis=-1) # 在最后一个维度（类别维度）上取最大值的索引
# print("Y_pred raw (0-based):", Y_pred) # 检查原始预测ID

# 将预测的ID（0-based）转换回字符
predicted_char_ids_raw = Y_pred[0] # 取第一个（也是唯一一个）样本的预测ID序列
# 修正：将0-based ID转回1-based的tokenizer索引
predicted_char_ids_actual = predicted_char_ids_raw + 1
# print("Y_pred adjusted (1-based):", predicted_char_ids_actual) # 检查调整后的ID

# 注意：Y_pred 是预测的下一个字符ID，需要将其转换回字符
# tokenizer.sequences_to_texts 接收的是 ID 列表，我们需要处理单个字符
# 修正：使用调整后的ID
predicted_chars = [tokenizer.index_word.get(id_, '<UNK>') for id_ in predicted_char_ids_actual]
# 使用 .get() 并提供默认值 '<UNK>' 以防万一ID仍然无效
generated_text = "".join(predicted_chars)
print(f"Predicted next character(s): {generated_text}")

# 如果你想像原代码那样只取最后一个预测的字符（即基于 "How are yo" 预测最后一个 'o' 的下一个字符）：
last_predicted_char_id_raw = predicted_char_ids_raw[-1] # 获取最后一个预测的原始ID (0-based)
last_predicted_char_id_actual = last_predicted_char_id_raw + 1 # 转换为实际ID (1-based)
last_predicted_char = tokenizer.index_word.get(last_predicted_char_id_actual, '<UNK>')
print(f"Predicted next character after 'yo': {last_predicted_char}")

1/1 [==============================] - 0s 14ms/step
Predicted next character(s): eu t e tou
Predicted next character after 'yo': u


In [15]:
#生成假莎士比亚文本

def next_char(text, temperature=1): 
    X_new = preprocess([text]) 
    y_proba = model.predict(X_new)[0, -1:, :] 
    rescaled_logits = tf.math.log(y_proba) / temperature 
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)+ 1 
    return tokenizer.sequences_to_texts(char_id.numpy())[0] 

def complete_text(text, n_chars=50, temperature=1): 
    for _ in range(n_chars): 
        text += next_char(text, temperature) 
    return text 

print(complete_text("w", temperature=1)) 

1/1 [==============================] - 0s 18ms/step
wars.

volumnia:
i show you.

cominius:
he drave he


In [ ]:
#有状态RNN
dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size]) 
dataset = dataset.window(window_length, shift=n_steps, drop_remain-der=True) 
dataset = dataset.flat_map(lambda window: window.batch(window_length)) 
dataset = dataset.batch(1) 
dataset = dataset.map(lambda windows: (windows[:, :-1], windows [:, 1:])) 
dataset = dataset.map( 
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=max_id), Y_batch)) 
dataset = dataset.prefetch(1) 

model = keras.models.Sequential([ 
    keras.layers.GR (128, return_sequences=True, stateful=True, 
                     dropout=0.2, recurrent_dropout=0.2, 
                     batch_input_shape=[batch_size, None, max_id]), 
    keras.layers.GR (128, return_sequences=True, stateful=True, 
                     dropout=0.2, recurrent_dropout=0.2), 
    keras.layers.TimeDistributed(keras.layers.Dense(max_id, 
                     activation="softmax")) 
]) 

class ResetStatesCallback(keras.callbacks.Callback): 
    def on_epoch_begin(self, epoch, logs): 
        self.model.reset_states() 
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam") 
model.fit(dataset, epochs=50, callbacks=[ResetStatesCallback()]) 